# Prediction of Bikecount

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb
from catboost import CatBoostRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

plt.style.use("tableau-colorblind10")

RANDOM_STATE=42

## Daten laden

In [2]:
df=pd.read_csv(r"../data/hour.csv")

## Preprocessing- und Feature-Engineering

In [3]:
class BasicPreprocessor(BaseEstimator, TransformerMixin):
    """Drop irrelevanter Spalten (instant, casual, registered) und Datums-Cast."""
    DROP_COLS = ["instant", "casual", "registered"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        if "dteday" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["dteday"]):
            df["dteday"] = pd.to_datetime(df["dteday"])
        drop = [c for c in self.DROP_COLS if c in df.columns]
        return df.drop(columns=drop)
    
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """OneHotEncoding für season, weathersit, mnth, hr, weekday."""
    CATEGORICAL_COLS = ["season", "weathersit", "mnth", "hr", "weekday"]

    def __init__(self):
        self.encoder_ = None
        self.cat_cols_present_ = None

    def fit(self, X, y=None):
        df = X.copy()
        self.cat_cols_present_ = [c for c in self.CATEGORICAL_COLS if c in df.columns]
        self.encoder_ = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        self.encoder_.fit(df[self.cat_cols_present_])
        return self

    def transform(self, X):
        df = X.copy()
        encoded = self.encoder_.transform(df[self.cat_cols_present_])
        encoded_df = pd.DataFrame(
            encoded,
            columns=self.encoder_.get_feature_names_out(self.cat_cols_present_),
            index=df.index,
        )
        return pd.concat([df.drop(columns=self.cat_cols_present_), encoded_df], axis=1)

class TimeSeriesFeatureEngineer(BaseEstimator, TransformerMixin):
    """Lag-, Rolling- und Differenz-Features."""
    LAGS    = [1, 2, 3, 6, 12, 24, 48]
    WINDOWS = [3, 6, 12, 24]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        if "dteday" in df.columns:
            df = df.sort_values("dteday").reset_index(drop=True)

        for lag in self.LAGS:
            df[f"cnt_lag_{lag}"] = df["cnt"].shift(lag)

        for window in self.WINDOWS:
            rolled = df["cnt"].shift(1).rolling(window)
            df[f"cnt_roll_mean_{window}"] = rolled.mean()
            df[f"cnt_roll_std_{window}"]  = rolled.std()
            df[f"cnt_roll_max_{window}"]  = rolled.max()

        df["cnt_diff_1"]  = df["cnt"].diff(1)
        df["cnt_diff_24"] = df["cnt"].diff(24)

        return df.dropna().reset_index(drop=True)

In [4]:
def build_pipeline() -> Pipeline:
    return Pipeline(steps=[
        ("basic",      BasicPreprocessor()),
        ("timeseries", TimeSeriesFeatureEngineer()),
        ("ohe",        CategoricalEncoder()),
    ])

In [5]:
pipeline=build_pipeline()
df=pipeline.fit_transform(df)
X=df.drop(columns="cnt")
y=df["cnt"]
df.head()

,dteday,yr,holiday,workingday,temp,atemp,hum,windspeed,cnt,cnt_lag_1,...,hr_21,hr_22,hr_23,weekday_0,weekday_1,weekday_2,weekday_3,weekday_4,weekday_5,weekday_6
0,2011-01-03,0,0,1,0.24,0.2121,0.35,0.2836,61,12.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2011-01-03,0,0,1,0.14,0.1515,0.69,0.1343,20,61.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,2011-01-03,0,0,1,0.18,0.1970,0.64,0.1343,52,20.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,2011-01-03,0,0,1,0.20,0.2273,0.47,0.1045,52,52.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,2011-01-03,0,0,1,0.20,0.2576,0.47,0.0000,110,52.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Train-Test-Split

In [ ]:
ts_outer = TimeSeriesSplit(n_splits=2, test_size=3400)
outer_splits = list(ts_outer.split(X))
train_idx, test_idx = outer_splits[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

ts_cv = TimeSeriesSplit(n_splits=5, test_size=1500)


## Evaluierung

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2':   r2_score(y_true, y_pred),
    }


def cross_validate_ts(model, X_tr, y_tr, cv):
    fold_maes = []
    for fold_train_idx, fold_val_idx in cv.split(X_tr):
        m = copy.deepcopy(model)
        m.fit(X_tr.iloc[fold_train_idx], y_tr.iloc[fold_train_idx])
        preds = m.predict(X_tr.iloc[fold_val_idx])
        fold_maes.append(mean_absolute_error(y_tr.iloc[fold_val_idx], preds))
    return np.mean(fold_maes), np.std(fold_maes)


def plot_predictions(r, y_test, n_plot=300):
    fig, ax = plt.subplots(figsize=(15, 3.5))
    y_true = y_test.values[:n_plot]
    n = min(n_plot, len(r['y_pred']))
    x_axis = range(n)
    ax.fill_between(x_axis, y_true[:n], alpha=0.15)
    ax.plot(x_axis, y_true[:n],      label='Wahrheit',   lw=1.2, alpha=0.8)
    ax.plot(x_axis, r['y_pred'][:n], label='Vorhersage', lw=1.0, alpha=0.9)
    ax.set_title(
        f"{r['name']}  |  MAE: {r['MAE']:.1f}  RMSE: {r['RMSE']:.1f}  R²: {r['R2']:.3f}",
        fontweight='bold', fontsize=11
    )
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylabel('cnt')
    ax.set_xlabel('Zeitschritt')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


all_results = []

## Modellierung
### Modellierung am Beispiel einer linearen Regression

### Modellierung am Beispiel eines HistGradientBoostingRegressor

### Modellierung am Beispiel eines RandomForestRegressors

### Modellierung am Beispiel von CatBoost

## Vergleich 